In [12]:
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic 
%pip install litellm
%pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ----- ---------------------------------- 2.1/16.2 MB 11.8 MB/s eta 0:00:02
   ----------- ---------------------------- 4.5/16.2 MB 11.7 MB/s eta 0:00:02
   ----------------- ---------------------- 7.1/16.2 MB 11.8 MB/s eta 0:00:01
   ----------------------- ---------------- 9.4/16.2 MB 11.7 MB/s eta 0:00:01
   ----------------------------- ---------- 12.1/16.2 MB 11.8 MB/s eta 0:00:01
   ----------------------------------- ---- 14.2/16.2 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------- 16.2/16.2 MB 11.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Set up environment variables

In [8]:
import os
import requests
from crewai.llm import LLM
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
load_dotenv()


C:\Users\edmar\AppData\Local\Temp\ipykernel_47824\1283091751.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


True

# Create Tavily Search tool

In [16]:
class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": os.getenv("TAVILY_API_KEY"),
            "query": query,
            "max_results": 5
        }

        response = requests.post(url, json=payload)
        data = response.json()

        results = []
        for r in data["results"]:
            results.append(f"{r['title']} - {r['url']}")

        return "\n".join(results)


search_tool = TavilySearchTool()


# Create FAISS Search tool

In [17]:
# Load FAISS vector store (this assumes you have already created and saved a FAISS index)
from langchain_openai import ChatOpenAI
from litellm import query


embeddings = OpenAIEmbeddings(
   
    api_version="2023-05-15",
    deployment="text-embedding-ada-002",
    api_key=os.environ["OPENAI_API_KEY"]
)

vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

# Initialize the gpt-5-mini model with temperature 0 for deterministic output

llm = ChatOpenAI(    
    model="gpt-5-mini",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"]
)

# Create a RetrievalQA chain that uses the retriever and LLM to answer queries with source context

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

class MondayAPIDocumentationSearchTool(BaseTool):
    name: str = "Monday.com API Search"
    description: str = "Search the Monday.com API for information."

    def _run(self, query: str):
        result = qa_chain.invoke({"query": query})
        return result["result"]        
    
search_tool = MondayAPIDocumentationSearchTool()

# Create Agents

In [27]:
agent_llm = LLM(
    model="gpt-5-mini",
    api_key=os.getenv("OPENAI_API_KEY")
)

router_agent = Agent(
    role="Request Router",
    goal="""
    Determine whether the user request is related to:
    - Monday.com api information
    OR
    - Other general information

    Return ONLY:
    MONDAY
    or
    OTHER
    """,
    backstory="Expert request classifier.",
    verbose=True,
)


monday_retriever = Agent(
    role="Monday API Researcher",
    goal="Find the answer to the user request  APusing the MondayAPIDocumentationSearchTool and provide relevant information.",
    backstory="You are an expert in searching the Monday.com API documentation and providing accurate information.",
    verbose=True,
    allow_delegation=False,
    llm=agent_llm,
    tools=[search_tool]
)

other_retriever = Agent(
    role="Web Searcher.",
    goal="Find the answer to the user request using the TavilySearchTool and provide relevant information.",
    backstory="You are an expert in web search and information retrieval.",
    verbose=True,
    allow_delegation=False,
    llm=agent_llm,
    tools=[search_tool]
)


# Define Tasks

In [30]:
user_request = input("Enter your request: ")
routing_task = Task(
    description=f"""
    Classify the following request:

    {user_request}

    Return only MONDAY or OTHER.
    """,
    expected_output="MONDAY or OTHER",
    agent=router_agent,
)

monday_retrieval_task = Task(
        description=f"""
        Retrieve search results for the following request from the Monday.com API documentation:

        {user_request}
        """,
        expected_output="Detailed Monday.com API information",
        agent=monday_retriever,
    )

tavily_retrieval_task = Task(
        description=f"""
        Retrieve search results for the following request using the TavilySearchTool:

        {user_request}
        """,
        expected_output="Detailed information from the web",
        agent=other_retriever,
    )

# Execute the request

In [31]:



# Execute router first
router_crew = Crew(
    agents=[router_agent],
    tasks=[routing_task],
    process=Process.sequential,
)

route = (await router_crew.akickoff()).raw.strip().upper()


print(f"Route selected: {route}")

# -------------------------
# Conditional Routing
# -------------------------

if route == "OTHER":
    crew = Crew(
        agents=[other_retriever],
        tasks=[tavily_retrieval_task],
        process=Process.sequential,
    )

elif route == "MONDAY":
    crew = Crew(
        agents=[monday_retriever],
        tasks=[monday_retrieval_task],
        process=Process.sequential,
    )

else:
    raise ValueError(f"Unknown route: {route}")

result = await crew.akickoff()

print(result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Request Router                                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Classify the following request:                                                                            │
│                                                                                                                 │
│      How do I retrieve a list of the column names in a Monday.com board                                         │
│                                                                                                                 │
│      Return only MONDAY or OTHER.                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Request Router                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  MONDAY                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Route selected: MONDAY


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Monday API Researcher                                                                                   │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Retrieve search results for the following request from the Monday.com API documentation:               │
│                                                                                                                 │
│          How do I retrieve a list of the column names in a Monday.com board                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool monday_com_api_search executed with result: You can get a board’s column names with the monday.com GraphQL API (v2). Use the boards(ids: ...) field and request the columns { id title ... } field. Required scope: boards:read.

Minimal GraphQL qu...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Monday API Researcher                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  You can get a board’s column names with the monday.com GraphQL API (v2). Use the boards(ids: ...) field and    │
│  request the columns { id title ... } field. Required scope: boards:read.                                       │
│                                                                                                                 │
│  Minimal GraphQL query                                                                                          │
│  query {                                                                                                        │
│    boards(ids: 123456789) {                                                                                     │
│      columns {                                                                                                  │
│        id                                                                                                       │
│        title                                                                                                    │
│      }                                                                                                          │
│    }                                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
│  curl example                                                                                                   │
│  curl -s -X POST https://api.monday.com/v2 \                                                                    │
│    -H "Authorization: <YOUR_API_TOKEN>" \                                                                       │
│    -H "Content-Type: application/json" \                                                                        │
│    -d '{"query":"query { boards(ids: 123456789) { columns { id title } } }"}'                                   │
│                                                                                                                 │
│  Node (fetch) example                                                                                           │
│  const fetch = require('node-fetch');                                                                           │
│  const BOARD_ID = 123456789;                                                                                    │
│  const query = `query { boards(ids: ${BOARD_ID}) { columns { id title } } }`;                                   │
│                                                                                                                 │
│  const res = await fetch('https://api.monday.com/v2', {                                                         │
│    method: 'POST',                                                                                              │
│    headers: {                                                                                                   │
│      'Authorization': process.env.MONDAY_TOKEN,                                                                 │
│      'Content-Type': 'application/json'                                                                         │
│    },                                                  

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

You can get a board’s column names with the monday.com GraphQL API (v2). Use the boards(ids: ...) field and request the columns { id title ... } field. Required scope: boards:read.

Minimal GraphQL query
query {
  boards(ids: 123456789) {
    columns {
      id
      title
    }
  }
}

curl example
curl -s -X POST https://api.monday.com/v2 \
  -H "Authorization: <YOUR_API_TOKEN>" \
  -H "Content-Type: application/json" \
  -d '{"query":"query { boards(ids: 123456789) { columns { id title } } }"}'

Node (fetch) example
const fetch = require('node-fetch');
const BOARD_ID = 123456789;
const query = `query { boards(ids: ${BOARD_ID}) { columns { id title } } }`;

const res = await fetch('https://api.monday.com/v2', {
  method: 'POST',
  headers: {
    'Authorization': process.env.MONDAY_TOKEN,
    'Content-Type': 'application/json'
  },
  body: JSON.stringify({ query })
});
const json = await res.json();
const columnNames = json.data.boards[0].columns.map(c => c.title);
console.log(columnNa